# Tutorial PXCT data analysis (Extra — Resolution estimation)

### Tutor: Julio C. da Silva (Néel Institute CNRS, Grenoble, France) 
### email: julio-cesar.da-silva@neel.cnrs.fr
#### Personal webpage: https://sites.google.com/view/jcesardasilva

### <span style="color:red">** Disclaimer: This notebook is intended for educational purposes only.**</span>
<span style="color:red">**Warning: You should have completed Parts 1, 2, and 3 before starting this Extra notebook.**</span>

<table class="tfo-notebook-buttons" align="center">
  <td>
    <a target="_blank" rel="noopener noreferrer" href="https://github.com/jcesardasilva/toupy"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

#### Importing packages again
Since we start a new notebook, we need to import the packages again:

In [ ]:
# standard packages
import sys
import time
# third party packages
import matplotlib.pyplot as plt
import numpy as np
import toupy

### Interactive backend

In [ ]:
%matplotlib widget

#### Let us reload our data 
We do this the same way we did in Part 2, but we only change the filename to `PXCTalignedprojections.npz`:

In [ ]:
fname = 'PXCTalignedprojections.npz'
data_dict = np.load(fname) # load the file
list(data_dict.files) # this one list the keys of the data dictionary extracted from the file
wavelen = data_dict['wavelen']
pixsize = data_dict['psize']
theta = data_dict['theta']
projections = data_dict['projections'] # <- ATTENTION: this one is memory consuming. 
nproj, nr, nc = projections.shape
delta_theta = np.diff(np.sort(theta))[0]

print(f"The total number of projections is {nproj}")
print(f"The angular sampling interval is {delta_theta:.02f} degrees")
print(f"The projection pixel size of the projections is {pixsize/1e-9:.02f} nm")
print(f"The wavelenth of the incoming photons is {wavelen/1e-10:.02f} Angstroms")

Let us take a look at one projection. I will select the first and last ones,i.e. at angles 0 and 180-$\Delta\theta$ degress:

## Extra — Spatial resolution estimation by Fourier Shell Correlation

The **Fourier Shell Correlation (FSC)** is the gold-standard method for estimating the resolution of a 3-D reconstruction. It was originally developed for cryo-electron microscopy but is equally applicable to X-ray tomography.

**Principle**: split the dataset into two independent half-datasets (odd/even projections), reconstruct two independent volumes (tomogram1, tomogram2), and compute the Fourier correlation between them shell by shell in reciprocal space:

$$\text{FSC}(k) = \frac{\sum_{\mathbf{q} \in \text{shell}} F_1(\mathbf{q})\, F_2^*(\mathbf{q})}{\sqrt{\sum |F_1|^2 \sum |F_2|^2}}$$

The resolution is defined as the spatial frequency where FSC falls below a threshold (typically the half-bit criterion). Beyond that frequency the two reconstructions are no longer correlated — i.e. the signal is buried in noise.

**Three methods are demonstrated**:
1. **FSC** — global resolution from the half-dataset correlation curve.
2. **RandomFSC** — phase-randomisation test to guard against overfitting.
3. **SSNR** (Spectral Signal-to-Noise Ratio) — an equivalent but more interpretable view of the same data.
4. **LocalFSC / LocalResolution** — produces a resolution *map* rather than a single number, revealing spatial heterogeneity.

> ⚠️ **This notebook is memory- and CPU-intensive** — it reconstructs two full volumes. Allow 15–30 minutes on a modern laptop.

## Advanced Mode: the spatial resolution estimate by Fourier Shell Correlation
In case you want to try to estimate the **spatial resolution** of the reconstructed slice, you can proceed to the next step. This part is memory and CPU-intensive and very time-consuming. 

<span style="color:red">**Proceed at your own responsibility**</span>

In [ ]:
from toupy.resolution import FSCPlot, compute_2tomograms_splitted, split_dataset
from toupy.utils import sort_array, tqdm
from toupy.tomo import tomo_recons

In [ ]:
# initializing params
params = dict()

# =========================
# Edit session
# =========================
params["slicenum"] = 200  # Choose the slice
params["limsyFSC"] = [100, 300]  # number of slices for the 3D FSC
params["filtertype"] = "hann"  # Filter to use for FBP
params["freqcutoff"] = 1.0  # Frequency cutoff (between 0 and 1)
params["circle"] = False #True
params["algorithm"] = "FBP"  # FBP or SART
params["derivatives"] = True  # only for FBP
params["calc_derivatives"] = False  # Calculate derivatives if not done
params["apod_width"] = 50  # apodization width in pixels
params["thick_ring"] = 4  # number of pixel to average each FRC ring
params["crop"] = None #[1465, 1865, 935, 1335]  # [top, bottom, left, right]
params["vmin_plot"] = None  # 0.5e-5
params["vmax_plot"] = None  # None
params["colormap"] = "bone"  # colormap to show images
# =========================

In [ ]:
# sorting theta
print("Sorting theta and projections accordingly.")
projections, theta = sort_array(projections, theta)
ntheta = theta.shape[0]
# convinient change of variables
slice_num = params["slicenum"]
vmin_plot = params["vmin_plot"]
vmax_plot = params["vmax_plot"]
limsyFSC = params["limsyFSC"]
nslices = limsyFSC[-1] - limsyFSC[0]
nprojs, nr, nc = projections.shape

In [ ]:
# initializing variables
tomogram1 = np.empty((nslices, nc, nc))
tomogram2 = np.empty((nslices, nc, nc))
sinogramempty = np.empty_like(np.transpose(projections[:, 0, :]))
sino1nr, sino1nc = sinogramempty.shape
sino2nr, sino2nc = sino1nr, sino1nc
sinogram1 = np.empty((nslices, sino1nr, int(sino1nc/2)))
sinogram2 = np.empty((nslices, sino2nr, int(sino2nc/2)))

In [ ]:
# splitting the sinograms
pbar = tqdm(range(limsyFSC[0], limsyFSC[-1]), desc="Sinogram for slice", colour='blue', file=sys.stdout)
for idx, ii in enumerate(pbar):
    pbar.set_description(f"Sinogram for slice {ii}")
    sinogram = np.transpose(projections[:, ii, :])
    sinogram1[idx], sinogram2[idx], theta1, theta2 = split_dataset(
        sinogram, theta
    )

In [ ]:
# calculating the 2 tomograms
pbar = tqdm(range(limsyFSC[0], limsyFSC[-1]), desc="Slice ", colour='blue', file=sys.stdout)
for idx, ii in enumerate(pbar):
    # dividing the data into two datasets and computing tomograms
    tomogram1[idx], tomogram2[idx] = compute_2tomograms_splitted(
        sinogram1[idx], sinogram2[idx], theta1, theta2, ** params
    )

In [ ]:
# 3D FSC
print("Estimating the resolution by 3D FSC...")
FSC3D = FSCPlot(
    tomogram1,
    tomogram2,
    threshold = "halfbit",
    ring_thick = params["thick_ring"],
    apod_width = params["apod_width"],
    pixel_size = pixsize
)
# Display the FSC curve
normfreqs, T, FSC3Dcurve, intersectFSC3D = FSC3D.plot() 

In [ ]:
print(f'The fraction of Nyquist: {FSC3D.fn_res:.04f}')           # e.g. 0.735  — fraction of Nyquist
print(f'The cycles/pixel: {FSC3D.fn_res_cpx:.04f}')       # e.g. 0.3675 — cycles/pixel
print(f'The voxelsize of the data is {pixsize/1e-9:.02f} nm')
print(f'The full period resolution: {FSC3D.resolution_full/1e-9:.02f} nm ({FSC3D.resolution_full/pixsize:.02f} voxels per resel)')  # e.g. 77.8e-9 m  (van Heel / FSC convention)
print(f'The half-period resolution: {FSC3D.resolution_half/1e-9:.02f} nm ({FSC3D.resolution_half/pixsize:.02f} voxels per resel)')  # e.g. 38.9e-9 m  (Rayleigh / feature-size convention)

### Testing the FSC results with Phase Randomization FSC — gold-standard guard against overfitting

In [ ]:
from toupy.resolution import RandomFSC
# First, inspect the FSC curve above to pick fsc_cutoff. 
# Then run the randomization test
rfsc = RandomFSC(tomogram1, tomogram2, threshold="halfbit", ring_thick=params["thick_ring"],
                apod_width=params["apod_width"], fsc_cutoff=0.8, pixel_size=pixsize)
# Plotting
fn, FSC_obs, FSC_rand, FSC_corr, T = rfsc.plot()
## Interpretation:
# If FSC_rand ≈ 0 beyond cutoff → no overfitting, your FSC result is genuine
# If FSC_rand stays elevated → overfitting/model bias is present, true resolution is
#   better read from FSC_corr than from FSC_obs
print(f"Phase randomization starts at shell {rfsc.cutoff_shell} "
      f"({rfsc.cutoff_shell / rfsc.fnyquist:.3f} × Nyquist)")

In [ ]:
# Resolution is evaluated on FSC_corr (the phase-corrected curve)
print(f'The full period resolution: {rfsc.resolution_full/1e-9:.02f} nm ({rfsc.resolution_full/rfsc.pixel_size:.02f} voxels per resel)')
print(f'The half-period resolution: {rfsc.resolution_half/1e-9:.02f} nm ({rfsc.resolution_half/rfsc.pixel_size:.02f} voxels per resel)')

## Another method (Based on the FSC results): Spectral SNR (SSNR) 

Resolution = frequency where SSNR drops below 1. It is an equivalent but more interpretable view of the same data: it expresses SNR per frequency shell instead of correlation.

In [ ]:
from toupy.resolution import SSNRPlot
ssnr = SSNRPlot(tomogram1, 
                tomogram2, 
                threshold="halfbit", 
                ring_thick=params["thick_ring"],
                apod_width=params["apod_width"],
                pixel_size = pixsize
               )
fn, FSC, SSNR, SSNR_T, fn_res_cpx= ssnr.plot()

In [ ]:
print(f'The fraction of Nyquist: {ssnr.fn_res:.04f}')           # e.g. 0.735  — fraction of Nyquist
print(f'The cycles/pixel: {ssnr.fn_res_cpx:.04f}')       # e.g. 0.3675 — cycles/pixel
print(f'The voxelsize of the data is {ssnr.pixel_size/1e-9:.02f} nm')
print(f'The full period resolution: {ssnr.resolution_full/1e-9:.02f} nm ({ssnr.resolution_full/ssnr.pixel_size:.02f} voxels per resel)')  # e.g. 77.8e-9 m  (van Heel / FSC convention)
print(f'The half-period resolution: {ssnr.resolution_half/1e-9:.02f} nm ({ssnr.resolution_half/ssnr.pixel_size:.02f} voxels per resel)')  # e.g. 38.9e-9 m  (Rayleigh / feature-size convention)

### Yet, another method: LocalFSC — local resolution map
Produces a resolution map rather than a global number — essential for heterogeneous tomographic specimens (biological tissue, composite materials)

In [ ]:
from toupy.resolution import LocalFSC

# Needs your two independent half-reconstructions
lfsc = LocalFSC(tomogram1, tomogram2, pixel_size=pixsize, box_size=32, step=16)
localFSCmaparray = lfsc.plot(slice_idx=100) 

In [ ]:
# median
print(f"Median local resolution (full period): {lfsc.resolution_median:.02f} px")
print(f"Median local resolution (half period): {lfsc.resolution_median_half:.02f} px")
# mean
print(f"Mean local resolution (full period): {lfsc.resolution_mean:.02f} px")
print(f"Mean local resolution (half period): {lfsc.resolution_mean_half:.02f} px")
# std
print(f"std of full period: {lfsc.resolution_std:.02f} px")
print(f"std of half period: {lfsc.resolution_std_half:.02f} px")
# voxel size and resolution
print(f'The voxelsize of the data is {lfsc.pixel_size/1e-9:.02f} nm')
print(f'The full period resolution: {lfsc.pixel_size*lfsc.resolution_mean/1e-9:.02f} nm ({lfsc.pixel_size*lfsc.resolution_mean/lfsc.pixel_size:.02f} voxels per resel)')  # e.g. 77.8e-9 m  (van Heel / FSC convention)
print(f'The half-period resolution: {lfsc.pixel_size*lfsc.resolution_mean_half/1e-9:.02f} nm ({lfsc.pixel_size*lfsc.resolution_mean_half/lfsc.pixel_size:.02f} voxels per resel)')  # e.g. 38.9e-9 m  (Rayleigh / feature-size convention)

### Local Resolution map

In [ ]:
from toupy.resolution import LocalResolution
# Only needs your single final reconstruction
# noise_method='corners' -> Standard Tomography (empty corners)
# noise_method='highfreq' -> Local Tomography (sample fills the FOV)
lr = LocalResolution(tomogram1, pixel_size=pixsize, significance=0.05,
                     n_freq=20, window_sigma=7.0, noise_method='corners')#noise_method='highfreq')

localresmap=lr.plot(slice_idx=150)   # saves LocalResolution_map.png

## Spatial maps
#lr.resolution_map            # full period, pixels
#lr.resolution_map_phys       # full period, physical units
#lr.resolution_map_half       # half period, pixels
#lr.resolution_map_phys_half  # half period, physical units

In [ ]:
print(f"Median local resolution (full period): {lr.resolution_median:.02f} px")
print(f"Mean local resolution (full period): {lr.resolution_mean:.02f} px")
print(f'The voxelsize of the data is {lr.pixel_size/1e-9:.02f} nm')
print(f'The full period resolution: {lr.pixel_size*lr.resolution_mean/1e-9:.02f} nm ({lr.pixel_size*lr.resolution_mean//lr.pixel_size:.02f} voxels per resel)')  # e.g. 77.8e-9 m  (van Heel / FSC convention)
print(f'The half-period resolution: {(lr.pixel_size/2)*lr.resolution_mean/1e-9:.02f} nm ({(lr.pixel_size/2)*lr.resolution_mean/lr.pixel_size:.02f} voxels per resel)')  # e.g. 38.9e-9 m  (Rayleigh / feature-size convention)

## Congratulations! You finished the tutorial with success.